# Patient-Aware DDI: Qwen3-14B LoRA
Project training and inference notebook, adapted from the Unsloth Qwen3 workflow.
Run in a CUDA GPU environment (for example Colab); CPU/macOS training is not supported.
Upload your own `synthetic_messages.jsonl` and `evaluation_dataset.jsonl` to the notebook working directory.
These datasets and trained adapters are not distributed in this repository.
This notebook records the experimental setup; its package installation cell is historical and may require adaptation for your CUDA runtime.
Use synthetic cases only. Outputs are research artifacts, not clinical recommendations.


Note to teacher: Created fine tuned weights can be downloaded at: 

https://drive.google.com/drive/folders/1QEMb_0svU6EHfC_M2mzL_mbaTmpYt2MW?usp=sharing

### Installation of the requirements

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Installing the Qwem3-14B Model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  
    lora_dropout = 0, 
    bias = "none",  
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Dataset

In [ ]:
import json
with open("synthetic_messages.jsonl", "r") as f:
    synthetic_messages = [json.loads(line) for line in f]

# Seeing an example message
print(synthetic_messages[0]['messages'][1])

In [ ]:
def parse_message(message: list):
    conversations = []
    for item in message:
        item = item['messages']

        conversations.append([{"role": "system", "content": item[0]['content']},
                              {"role": "user", "content": item[1]['content']}, 
                          {"role": "assistant", "content": item[2]['content']}])
    return conversations

In [ ]:
ddi_dataset = parse_message(synthetic_messages)

In [ ]:
ddi_conversations = tokenizer.apply_chat_template(
    ddi_dataset,
    tokenize = False,
)

Let's see the first transformed row usable for model fine-tuning:

In [ ]:
ddi_conversations[0]

Unsloth requires dataset to be in pandas format. So we are converting to panas here.

In [ ]:
import pandas as pd

data = pd.concat([
    pd.Series(ddi_conversations),
])
data.name = "text"

from datasets import Dataset
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = 3407)

### Training

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        padding_free  = False, # Set to True if > 17 GB VRAM
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
ddi_dataset[0]

In [ ]:
import json 
with open("evaluation_dataset.jsonl", "r") as f:
    evaluation_dataset = [json.loads(line) for line in f]

print(evaluation_dataset[0]['rewrited_messages'][1])



Small test being made

In [ ]:
messages = [
    {"role" : "system", "content" : "You are a cautious educational clinical decision-support assistant. Use only source-supported reasoning, avoid unsupported claims, and clearly state uncertainty when evidence is insufficient. This is synthetic educational data and not medical advice."},
    {"role" : "user", "content" : "fentanyl and alcohol together, 20 years old, terminal illness, diabetes, heroin addiction"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
evaluation_dataset[0]['rewrited_messages']
print(evaluation_dataset[0]['rewrited_messages'][1])

In [ ]:
from html import escape
from IPython.display import HTML, display
from tqdm.notebook import tqdm

system_prompt = evaluation_dataset[0]["rewrited_messages"][0]["content"]
model_outputs = []

latest_display = display(
    HTML("<b>Waiting for first generation...</b>"),
    display_id=True,
)

print("System prompt:", system_prompt)

for idx, message in enumerate(
    tqdm(evaluation_dataset, desc="Generating answers", unit="sample"),
    start=1,
):
    test_data_msg = message["rewrited_messages"][1]["content"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": test_data_msg},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    model_outputs.append({
        "index": idx - 1,
        "metadata": message.get("metadata", {}),
        "user": test_data_msg,
        "generated_text": generated_text,
    })

    latest_display.update(HTML(f"""
    <h4>Latest generation ({idx}/{len(evaluation_dataset)})</h4>

    <b>User query</b>
    <pre style="white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;">{escape(test_data_msg)}</pre>

    <b>Generated answer</b>
    <pre style="white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;">{escape(generated_text)}</pre>
    """))

print(f"Done. Generated {len(model_outputs)} answers.")

In [ ]:
import json
from pathlib import Path

output_path = Path("sample_outputs_finetuned_2.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for line in model_outputs:
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

In [ ]:
print(model_outputs[0])